In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch, triton
assert sys.version.split()[0] == '3.13.15'
assert np.__version__ == '2.3.5'
assert torch.__version__ == '2.11.0+cu128'
assert torch.version.cuda == '12.8'
assert triton.__version__ == '3.6.0'
assert len(os.sched_getaffinity(0)) == 4
assert torch.cuda.device_count() == 1
assert torch.cuda.get_device_name(0) == 'Tesla T4'
assert os.getuid() == 10001
assert not Path('/var/run/docker.sock').exists()
assert not Path('/etc/soup-queue/worker.env').exists()
assert not any('TOKEN' in k and k != 'TOKENIZERS_PARALLELISM' for k in os.environ)
assert Path('/sys/fs/cgroup/memory.max').read_text().strip() == str(30 * 1024**3)
assert Path('/sys/fs/cgroup/memory.swap.max').read_text().strip() == '0'
assert Path('/sys/fs/cgroup/cpuset.cpus.effective').read_text().strip() == '0-3'
print(sys.version, torch.__version__, triton.__version__, torch.cuda.get_device_name(0))


In [ ]:
%time sum(range(10000))
!pwd


In [ ]:
@torch.compile
def square_plus_one(x):
    return x.square() + 1
x = torch.arange(1024, device='cuda', dtype=torch.float32)
y = square_plus_one(x)
torch.testing.assert_close(y, x.square() + 1)
a = np.arange(8, dtype=np.float32)
np.testing.assert_array_equal(torch.from_numpy(a).cuda().cpu().numpy(), a)
result = {'passed': True, 'gpu': torch.cuda.get_device_name(0), 'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'triton': triton.__version__, 'numpy': np.__version__, 'cpus': sorted(os.sched_getaffinity(0)), 'memory_limit_gib': 30}
Path(os.environ['SOUP_RESULTS'], 'gpu-smoke.json').write_text(json.dumps(result, indent=2))
print('PASS: compiled GPU execution, NumPy round-trip, notebook magics, and isolation checks')
